In [3]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('new_heart_rates.csv')

# Display the first few rows of the dataframe
df.head()

,user_id,heart_rates
0,10921915,"[100, 111, 120, 119, 120, 116, 125, 128, 131, ..."
1,10921915,"[100, 105, 111, 110, 108, 115, 126, 130, 132, ..."
2,10921915,"[99, 105, 113, 110, 109, 110, 108, 121, 116, 1..."
3,10921915,"[99, 105, 113, 109, 112, 116, 116, 114, 114, 1..."
4,10921915,"[110, 113, 114, 116, 123, 126, 129, 135, 137, ..."


In [6]:
df["heart_rates"] = df["heart_rates"].apply(eval)

In [7]:
df.head()

,user_id,heart_rates
0,10921915,"[100, 111, 120, 119, 120, 116, 125, 128, 131, ..."
1,10921915,"[100, 105, 111, 110, 108, 115, 126, 130, 132, ..."
2,10921915,"[99, 105, 113, 110, 109, 110, 108, 121, 116, 1..."
3,10921915,"[99, 105, 113, 109, 112, 116, 116, 114, 114, 1..."
4,10921915,"[110, 113, 114, 116, 123, 126, 129, 135, 137, ..."


In [10]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 5.8 MB/s eta 0:00:005.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 5.2 MB/s eta 0:00:004.8 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [11]:
from sklearn.preprocessing import MinMaxScaler

In [14]:
df["heart_rates"]

0       [100, 111, 120, 119, 120, 116, 125, 128, 131, ...
1       [100, 105, 111, 110, 108, 115, 126, 130, 132, ...
2       [99, 105, 113, 110, 109, 110, 108, 121, 116, 1...
3       [99, 105, 113, 109, 112, 116, 116, 114, 114, 1...
4       [110, 113, 114, 116, 123, 126, 129, 135, 137, ...
                              ...                        
9195    [72, 119, 112, 123, 129, 132, 129, 133, 141, 1...
9196    [83, 79, 77, 79, 82, 81, 80, 80, 79, 78, 84, 8...
9197    [65, 76, 93, 98, 97, 88, 104, 110, 113, 108, 1...
9198    [78, 77, 76, 85, 91, 93, 96, 99, 99, 109, 121,...
9199    [99, 102, 106, 109, 111, 113, 115, 116, 116, 1...
Name: heart_rates, Length: 9200, dtype: object

In [ ]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [19]:
import numpy as np

In [25]:
scalers = {}

# Function to normalize heart rate lists
def normalize_user_data(user_df):
    scaler = MinMaxScaler()
    
    # Flatten the heart rate data for the user
    all_heart_rates = np.concatenate(user_df["heart_rates"].values).reshape(-1, 1)
    
    # Fit and transform the data
    normalized_hr = scaler.fit_transform(all_heart_rates).flatten()
    
    # Store the scaler for later use
    scalers[user_df["user_id"].iloc[0]] = scaler

    # Split back the normalized values into original lists
    split_indices = np.cumsum([len(l) for l in user_df["heart_rates"].values])[:-1]
    normalized_lists = np.split(normalized_hr, split_indices)
    
    return pd.Series(normalized_lists, index=user_df.index)

In [28]:
df["normalized_heart_rates"] = df.groupby("user_id", group_keys=False).apply(normalize_user_data)

/var/folders/nt/t40yshr55ll6r_nxryvqgf4r0000gn/T/ipykernel_10369/3300931582.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["normalized_heart_rates"] = df.groupby("user_id", group_keys=False).apply(normalize_user_data)


In [30]:
df.head()

,user_id,heart_rates,normalized_heart_rates
0,10921915,"[100, 111, 120, 119, 120, 116, 125, 128, 131, ...","[0.1826923076923077, 0.28846153846153844, 0.37..."
1,10921915,"[100, 105, 111, 110, 108, 115, 126, 130, 132, ...","[0.1826923076923077, 0.23076923076923073, 0.28..."
2,10921915,"[99, 105, 113, 110, 109, 110, 108, 121, 116, 1...","[0.17307692307692313, 0.23076923076923073, 0.3..."
3,10921915,"[99, 105, 113, 109, 112, 116, 116, 114, 114, 1...","[0.17307692307692313, 0.23076923076923073, 0.3..."
4,10921915,"[110, 113, 114, 116, 123, 126, 129, 135, 137, ...","[0.27884615384615385, 0.3076923076923078, 0.31..."


In [31]:
# Define sequence length (e.g., 50 time steps per input)
SEQ_LENGTH = 50

# Function to generate sequences from a continuous list of heart rates
def create_sequences(hr_list, seq_length):
    sequences = []
    for i in range(len(hr_list) - seq_length + 1):
        sequences.append(hr_list[i : i + seq_length])  # Extract window
    return np.array(sequences)

# Create sequences per user and stack
X_train = []

for _, user_df in df.groupby("user_id"):
    user_hr = np.concatenate(user_df["normalized_heart_rates"].values)  # Flatten per user
    user_sequences = create_sequences(user_hr, SEQ_LENGTH)
    X_train.append(user_sequences)

X_train = np.vstack(X_train)  # Combine all users
X_train = np.expand_dims(X_train, axis=-1)  # Add feature dimension for LSTM (batch, seq_len, 1)

print(f"Final shape of X_train: {X_train.shape}")  # Expected: (num_samples, SEQ_LENGTH, 1)


Final shape of X_train: (4597354, 50, 1)


In [33]:
%pip install torch

  Using cached typing_extensions-4.12.2-py3-none-any.whl.metadata (3.0 kB)
  Using cached setuptools-75.8.2-py3-none-any.whl.metadata (6.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 MB 2.9 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 3.7 MB/s eta 0:00:003.8 MB/s eta 0:00:01
Using cached typing_extensions-4.12.2-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 4.0 MB/s eta 0:00:004.0 MB/s eta 0:00:01
Using cached setuptools-75.8.2-py3-none-any.whl (1.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 5.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [34]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define LSTM Autoencoder
class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, latent_dim=32):
        super(LSTMAutoencoder, self).__init__()
        self.encoder = nn.LSTM(input_size=n_features, hidden_size=64, num_layers=2, batch_first=True)
        self.latent = nn.Linear(64, latent_dim)  # Bottleneck layer
        self.decoder = nn.LSTM(input_size=latent_dim, hidden_size=64, num_layers=2, batch_first=True)
        self.output_layer = nn.Linear(64, n_features)

    def forward(self, x):
        # Encoder
        encoded, _ = self.encoder(x)
        latent = self.latent(encoded[:, -1, :])  # Take last time step output

        # Repeat latent representation for each time step in decoder
        latent_repeated = latent.unsqueeze(1).repeat(1, x.shape[1], 1)

        # Decoder
        decoded, _ = self.decoder(latent_repeated)
        reconstructed = self.output_layer(decoded)

        return reconstructed

# Define model
seq_len = 50  # Time steps
n_features = 1  # Single feature (heart rate)
model = LSTMAutoencoder(seq_len, n_features)


In [51]:
from torch.utils.data import DataLoader, TensorDataset

# Convert NumPy array to PyTorch Tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)

# Create DataLoader for efficient batching
train_dataset = TensorDataset(X_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)


In [52]:
# Define loss function & optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Move model to GPU if available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)


LSTMAutoencoder(
  (encoder): LSTM(1, 64, num_layers=2, batch_first=True)
  (latent): Linear(in_features=64, out_features=32, bias=True)
  (decoder): LSTM(32, 64, num_layers=2, batch_first=True)
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
)

In [53]:
device

device(type='mps')

In [54]:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    print(f"Epoch {epoch+1} {len(train_loader)}")
    i = 0
    for batch in train_loader:
        batch = batch[0].to(device)
        optimizer.zero_grad()
        reconstructed = model(batch)
        loss = criterion(reconstructed, batch)  # Compare reconstruction with original
        loss.backward()
        optimizer.step()
        print(f"epoch {epoch+1} batch {i+1}")
        epoch_loss += loss.item()
        i += 1

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss / len(train_loader):.6f}")

Epoch 1 17959
epoch 1 batch 1
epoch 1 batch 2
epoch 1 batch 3
epoch 1 batch 4
epoch 1 batch 5
epoch 1 batch 6
epoch 1 batch 7
epoch 1 batch 8
epoch 1 batch 9
epoch 1 batch 10
epoch 1 batch 11
epoch 1 batch 12
epoch 1 batch 13
epoch 1 batch 14
epoch 1 batch 15
epoch 1 batch 16
epoch 1 batch 17
epoch 1 batch 18
epoch 1 batch 19
epoch 1 batch 20
epoch 1 batch 21
epoch 1 batch 22
epoch 1 batch 23
epoch 1 batch 24
epoch 1 batch 25
epoch 1 batch 26
epoch 1 batch 27
epoch 1 batch 28
epoch 1 batch 29
epoch 1 batch 30
epoch 1 batch 31
epoch 1 batch 32
epoch 1 batch 33
epoch 1 batch 34
epoch 1 batch 35
epoch 1 batch 36
epoch 1 batch 37
epoch 1 batch 38
epoch 1 batch 39
epoch 1 batch 40
epoch 1 batch 41
epoch 1 batch 42
epoch 1 batch 43
epoch 1 batch 44
epoch 1 batch 45
epoch 1 batch 46
epoch 1 batch 47
epoch 1 batch 48
epoch 1 batch 49
epoch 1 batch 50
epoch 1 batch 51
epoch 1 batch 52
epoch 1 batch 53
epoch 1 batch 54
epoch 1 batch 55
epoch 1 batch 56
epoch 1 batch 57
epoch 1 batch 58
epoch 1 b

In [56]:
torch.save(model.state_dict(), "heart_rate_predictor_lstm_model.pth")

In [62]:
%pip install torch.onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 6.4 MB/s eta 0:00:00 MB/s eta 0:00:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached protobuf-5.29.3-cp38-abi3-macosx_10_9_universal2.whl.metadata (592 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.5/693.5 kB 8.2 MB/s eta 0:00:00
Using cached protobuf-5.29.3-cp38-abi3-macosx_10_9_universal2.whl (417 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.7/667.7 kB 7.5 MB/s eta 0:00:00
  Created wheel for onnx: filename=onnx-1.17.0-cp313-cp313-macosx_14_0_arm64.whl size=15681252 sha256=1a1cd61b4463606937961dc5cb24ad90a77bfabf9d79dd41c0fd268dfe904a10
  Stored in directory: /Users/apple/Library/Caches/pip/wheels/ac/c1/73/4fa9bcde707160b22aa12f1d86d447ee5de075399d45f566e9
Successfully built onnx
Note: you may need to restart the kernel to use updated packages.


In [63]:
import torch.onnx

dummy_input = torch.randn(1, 50, 1).to(device)  # (batch_size=1, seq_len=50, features=1)
torch.onnx.export(model, dummy_input, "lstm_model.onnx", 
                  export_params=True, input_names=["input"], output_names=["output"])


/Users/apple/projects/digital-twin/.venv/lib/python3.13/site-packages/torch/onnx/symbolic_opset9.py:4277: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(
